In [1]:
# !pip install transformers pandas tqdm

In [1]:
import subprocess
import os

# result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
# output = result.stdout
# for line in output.splitlines():
#     if '=' in line:
#         var, value = line.split('=', 1)
#         os.environ[var] = value

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
import torch
import pandas as pd
from transformers import T5ForSequenceClassification, T5Tokenizer
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.nn.functional import cross_entropy
from torch import nn
from tqdm.auto import tqdm

In [8]:
# Define

# chinese - ctb / english - ptb / korean - ktb

middle_file_name = "ptb"

# withlabels - "" / withoutlabels - ".withoutlabels"

end_file_name = ""
# end_file_name = ".withoutlabels"

train_source_path = "train." + middle_file_name + ".source.linearized" + end_file_name
train_target_path = "train." + middle_file_name + ".target.linearized" + end_file_name
dev_source_path = "dev." + middle_file_name + ".source.linearized" + end_file_name
dev_target_path = "dev." + middle_file_name + ".target.linearized" + end_file_name
test_source_path = "test." + middle_file_name + ".source.linearized" + end_file_name
test_target_path = "test." + middle_file_name + ".target.linearized" + end_file_name

prediction_output_path = "test." + middle_file_name + ".predict.linearized" + end_file_name

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_small_path = "/root/autodl-fs/huggingface/hub/models--google-t5--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4"
tokenizer = AutoTokenizer.from_pretrained(t5_small_path)
model = AutoModelForSeq2SeqLM.from_pretrained(t5_small_path)
# tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-small")
# model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-small")


In [10]:
def read_and_tokenize(file_path, tokenizer):
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f.readlines()]
    return tokenizer(lines, max_length=512, padding="max_length", truncation=True, return_tensors="pt")

# Reading and tokenizing data
train_source = read_and_tokenize(train_source_path, tokenizer)
train_target = read_and_tokenize(train_target_path, tokenizer)
dev_source = read_and_tokenize(dev_source_path, tokenizer)
dev_target = read_and_tokenize(dev_target_path, tokenizer)
test_source = read_and_tokenize(test_source_path, tokenizer)
test_target = read_and_tokenize(test_target_path, tokenizer)

# Custom dataset class
class TreeDataset(Dataset):
    def __init__(self, encodings, targets):
        self.encodings = encodings
        self.targets = targets['input_ids'] 

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.targets[idx]
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# Create datasets
train_dataset = TreeDataset(train_source, train_target)
dev_dataset = TreeDataset(dev_source, dev_target)
test_dataset = TreeDataset(test_source, test_target)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [12]:
# Define the optimizer and loss function
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = cross_entropy

In [13]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    progress_bar = tqdm(data_loader, desc="Training", leave=False)
    for batch in progress_bar:
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(**batch)
        
        # Backward pass
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        
        # Update parameters and zero gradients
        optimizer.step()
        optimizer.zero_grad()

        # Update progress bar
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(data_loader)

In [14]:
# Function to evaluate the model
def eval_model(model, data_loader, tokenizer, device):
    model.eval()
    total_matches = 0
    total_samples = 0
    progress_bar = tqdm(data_loader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for batch in progress_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model.generate(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], max_new_tokens=3500)
            
            # Decode predictions
            predictions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
            references = [tokenizer.decode(batch['labels'][i], skip_special_tokens=True) for i in range(len(batch['labels']))]
            
            # Calculate exact matches
            matches = sum([1 for pred, ref in zip(predictions, references) if pred == ref])
            total_matches += matches
            total_samples += len(batch['labels'])
            
            # Update progress bar
            progress_bar.set_postfix({'accuracy': f'{(matches / len(batch["labels"])):.4f}'})
    
    return total_matches / total_samples


In [15]:
# Training and evaluation loop
num_epochs = 10
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_loss = train_epoch(model, train_loader, optimizer, device)
    print(f"Training loss: {train_loss:.4f}")
    
    dev_loss = eval_model(model, dev_loader, tokenizer, device)
    print(f"Validation loss: {dev_loss:.4f}")

Epoch 1/10


Training:   0%|          | 0/2490 [00:00<?, ?it/s]

Training loss: 0.1039


Evaluating:   0%|          | 0/107 [00:00<?, ?it/s]

Validation loss: 0.6165
Epoch 2/10


...

Validation loss: 0.8206
Epoch 10/10


Training:   0%|          | 0/2490 [00:00<?, ?it/s]

Training loss: 0.0023


Evaluating:   0%|          | 0/107 [00:00<?, ?it/s]

Validation loss: 0.8265


In [16]:
def predict(input_text, model, tokenizer, device, max_new_tokens=3000):
    # Tokenize the input text
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)  # Ensure input tensor is on the correct device

    # Generate the output with specified maximum new tokens
    output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens)

    # Decode the output
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return output_text


In [13]:
import time

def predict_and_evaluate(test_source, test_target, model, tokenizer, device, output_file_path):
    # Prepare the output file and initialize metrics
    with open(output_file_path, 'w') as f:
        total_correct = 0
        total_samples = len(test_source)
        
        # Measure start time
        start_time = time.time()
        
        # Process each line in the test source
        for input_text, target_text in tqdm(zip(test_source, test_target), total=len(test_source), desc="Processing"):
            # Tokenize and predict
            input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
            output_ids = model.generate(input_ids, max_new_tokens=3500)  
            prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            
            # Write prediction to file
            f.write(prediction + "\n")
            
            # Check if the prediction is correct
            if prediction.strip() == target_text.strip():
                total_correct += 1

        # Measure end time
        end_time = time.time()
    
    # Calculate the total time taken and accuracy
    total_time = end_time - start_time
    accuracy = total_correct / total_samples if total_samples > 0 else 0
    return total_time, accuracy

def read_linearized_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    return lines
    
# Assuming the model, tokenizer, and device are already set up
output_file_path = prediction_output_path
test_source = read_linearized_file(test_source_path)
test_target = read_linearized_file(test_target_path)

# Run the prediction and evaluation
prediction_time, accuracy = predict_and_evaluate(test_source, test_target, model, tokenizer, device, output_file_path)

print(f"Predictions saved to {output_file_path}")
print(f"Total prediction time: {prediction_time:.2f} seconds")
print(f"Exact Match Accuracy on Test Set: {accuracy:.4f}")


Processing:   0%|          | 0/348 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (739 > 512). Running this sequence through the model will result in indexing errors


Predictions saved to test.ctb.predict.linearized.withoutlabels
Total prediction time: 267.84 seconds
Exact Match Accuracy on Test Set: 0.8046


In [17]:
torch.save(model, '10epoch_english_with_T5small.pth')

In [19]:
# Output new prediction
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


new_test_source_path = "./newIO/test.ptb.neural.wout.nulls.with.labs"
new_test_source = read_and_tokenize(new_test_source_path, tokenizer)
new_test_output_path = new_test_source_path+"_output"
# model = ""


with open(new_test_output_path, "w") as f:
    for input_text in new_test_source:
        prediction = predict(input_text, model, tokenizer, device)
        f.write(prediction + "\n")

print("Save to predictions")

Save to predictions


In [21]:
import time
def new_predict(test_source, model, tokenizer, device, output_file_path):
    with open(output_file_path, 'w') as f:
        start_time = time.time()
        
        for input_text in tqdm(test_source, desc="Processing"):
            input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
            output_ids = model.generate(input_ids, max_new_tokens=3500)  
            prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            
            f.write(prediction + "\n")
        
        end_time = time.time()
    
    total_time = end_time - start_time
    return total_time

def read_linearized_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    return lines


model.to(device)


test_source_path = "./newIO/test.ptb.neural.wout.nulls.with.labs"
output_file_path = "./newIO/test.ptb.neural.wout.nulls.with.labs_output" 


test_source = read_linearized_file(test_source_path)


prediction_time = new_predict(test_source, model, tokenizer, device, output_file_path)

print(f"Predictions saved to {output_file_path}")
print(f"Total prediction time: {prediction_time:.2f} seconds")

Processing:   0%|          | 0/2416 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (527 > 512). Running this sequence through the model will result in indexing errors


Predictions saved to ./newIO/test.ptb.neural.wout.nulls.with.labs_output
Total prediction time: 2634.46 seconds
